In [ ]:
import os
import numpy as np
import tensorflow as tf


from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

In [ ]:
#cnn feature extraction
cnn=VGG16(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

cnn.trainable=False

In [ ]:
#mock input, in future we will loop this over the entire directory to process all 100 signs
sequence_path=(r"C:\Users\darsh\Downloads\archive\preprocessing\val\pose\woman\68764")

image_paths=sorted([
    os.path.join(sequence_path,filename)
    for filename in os.listdir(sequence_path)
    if filename.lower().endswith(".jpg")
])

In [ ]:
#load frames
frames=[]

for image_path in image_paths:
    image=tf.keras.utils.load_img(
        image_path,
        target_size=(224,224)
    )

    image=tf.keras.utils.img_to_array(image)

    frames.append(image)

frames=np.array(frames)

print("Frames shape: ",frames.shape)

Frames shape:  (16, 224, 224, 3)


In [ ]:
#cnn preprocessing
frames=preprocess_input(frames)

#extract features
features=cnn.predict(frames)

print("Feature sequence shapes: ",features.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step
Feature sequence shapes:  (16, 512)


In [ ]:
#gru
gru=tf.keras.layers.GRU(128)

#inputs
inputs=tf.keras.Input(shape=(None, 512))

gru_output=gru(inputs)

print("GRU output shape: ",gru_output.shape)

#output
outputs=tf.keras.layers.Dense(
    100,
    activation="softmax"
)(gru_output)

model=tf.keras.Model(inputs=inputs, outputs=outputs)

model.summary()

GRU output shape:  (None, 128)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)           │ (None, None, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru_5 (GRU)                          │ (None, 128)                 │         246,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 100)                 │          12,900 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 259,428 (1013.39 KB)

 Trainable params: 259,428 (1013.39 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#iterating through the dataset

import os

#dataset path
DATASET_ROOT=r"C:\Users\darsh\Documents\ML_Datasets\wlasl\preprocessing"

dataset=[]

for split in ["train", "val", "test"]:

    print(f"\n========== {split.upper()} ==========")

    frames_path=os.path.join(
        DATASET_ROOT,
        split,
        "frames"
    )

    for word in os.listdir(frames_path):
        word_path=os.path.join(
            frames_path,
            word
        )

        if not os.path.isdir(word_path):
            continue

        for sequence in os.listdir(word_path):
            sequence_path=os.path.join(
                word_path,
                sequence
            )

            if not os.path.isdir(sequence_path):
                continue

            #all image paths
            image_paths=[]

            for image in sorted(os.listdir(sequence_path)):
                if image.lower().endswith(".jpg"):
                    image_path=os.path.join(
                        sequence_path,
                        image
                    )

                image_paths.append(image_path)

            dataset.append({
                "split":split,
                "word":word,
                "sequence":sequence,
                "image_paths":image_paths
            })

print("Total sequence: ",len(dataset))

print(dataset[0])


In [ ]:
import tensorflow as tf
import numpy as np
def load_sequence_images(image_paths):
    frames=[]

    for image in image_paths:
        image=tf.keras.utils.load_img(
            image_path,
            target_size=(224,224)
        )

        image=tf.keras.utils.img_to_array(image)

        frames.append(image)

    return np.array(frames, dtype=np.float32)


# sample=dataset[0]

# frames=load_sequence_images(
#     sample["image_paths"]
# )

# print("word:",sample["word"])
# print("sequence:",sample["sequence"])
# print("Frames shapes:",frames.shape)


In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input

def extract_cnn_features(image_paths):

    #load images
    frames=load_sequence_images(
        image_paths
    )

    #preprocess for VGG16
    frames=preprocess_input(frames)

    #extract features
    features=cnn.predict(
        frames,
        verbose=0
    )

    return features

In [ ]:
def process_split(split_name):
    X=[]
    y=[]

    for item in dataset:

        if item["split"]!=split_name:
            continue

        features=extract_cnn_features(
            item["image_paths"]
        )

        #store features
        X.append(features)

        #store word label
        y.append(item["word"])

    return X,y

In [ ]:
#this is just for demonstration, to test one sequence only

from tensorflow.keras.applications.vgg16 import preprocess_input

# Select one sequence
sample = dataset[0]

# Load images
frames = load_sequence_images(
    sample["image_paths"]
)

# Preprocess images for VGG16
frames_preprocessed = preprocess_input(frames)

# Extract CNN features
features = cnn.predict(
    frames_preprocessed,
    verbose=0
)

# Display results
print("Word:", sample["word"])
print("Sequence:", sample["sequence"])
print("Frames shape:", frames.shape)
print("Features shape:", features.shape)